## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 53 samples and 11449 columns.
First few rows of the data:


,OS (month),Vital status,DFS (month),Disease-free,Sample,Op,Pathology,Risk factor,Progression,Tumor stage,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,46,0,46,0,DH013,TH,DN high,HBV,NaN,Stage II,...,7.452408,3.620973,6.167827,8.543239,5.840271,5.237661,6.757061,6.641830,4.346114,6.949632
1,102,0,102,0,TG1-003,TH,ES1,HBV,NaN,Stage II,...,7.336706,3.639574,6.317031,7.831987,5.338136,5.270831,6.924206,6.534365,3.967166,6.925958
2,119,0,119,0,TG1-005,TH,ES1,HBV,NaN,Stage II,...,6.934483,3.769259,6.098319,7.832153,5.097037,5.028956,6.750284,6.649036,4.280020,6.924831
3,91,0,91,0,TG1-007,TH,ES1,HCV,NaN,Stage II,...,7.091694,3.916144,6.154706,7.928919,5.603064,5.310771,6.782520,6.695312,4.229381,7.131261
4,84,0,84,0,TG1-009,TH,ES1,HBV,NaN,Stage I,...,7.750944,4.205411,6.733334,8.155534,6.379199,5.685522,7.172040,6.495033,4.414602,6.813365


## Load gene sets

In [2]:
MSIGDB_JSON = Path("h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
